In [29]:
from utils import * 

%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
contig_metadata_df = pd.read_csv('../data/blast/ggkbase/blast_ggkbase_contig_metadata.csv') # This has not been clustered and dereplicated. 
contig_metadata_df = contig_metadata_df[contig_metadata_df.location == 'serpens_ridge_soil']

filter_ = lambda record : record.id in contig_metadata_df.contig_id.values
contigs_df = FASTAFile.from_file('../data/blast/ggkbase/blast_ggkbase_contigs.fasta', filter_=filter_).to_df()
contigs_df = contigs_df.drop_duplicates('seq')

contigs_df['contig_id'] = contigs_df.index 
contigs_df = contigs_df.merge(contig_metadata_df, on='contig_id', how='left')
contigs_df['length'] = contigs_df.seq.apply(len)
contigs_df = contigs_df.sort_values('length', ascending=False)
contigs_df['project_id'] = [contig_id.split('_scaffold_')[0] for contig_id in contigs_df.contig_id]



In [31]:
genomes = dict()
# Checked all contigs from this sample, and seems as though these four are the main candidates. 
genomes['SR-VP_07_25_2022_A1_115cm_MG_illumina_7008_11933_23519_93193'] = ['SR-VP_07_25_2022_A1_115cm_MG_illumina_7008', 'SR-VP_07_25_2022_A1_115cm_MG_illumina_11933', 'SR-VP_07_25_2022_A1_115cm_MG_illumina_23519', 'SR-VP_07_25_2022_A1_115cm_MG_illumina_93193']
genomes['SRVP19_trench_A_40cm-2_viral_idba_scaffold_30523'] = ['SRVP19_trench_A_40cm-2_viral_idba_scaffold_30523']
genomes['SR-VP_07_25_2022_A1_60cm_MG_illumina_5472_8855_31448'] = ['SR-VP_07_25_2022_A1_60cm_MG_illumina_5472', 'SR-VP_07_25_2022_A1_60cm_MG_illumina_8855', 'SR-VP_07_25_2022_A1_60cm_MG_illumina_31448']
genomes['SR-VP_9_9_2021_56_4A_0_95m_scaffold_3886_5485'] = ['SR-VP_9_9_2021_56_4A_0_95m_scaffold_5485', 'SR-VP_9_9_2021_56_4A_0_95m_scaffold_3886']

project_id_map = dict()
project_id_map['SR-VP_07_25_2022_A1_115cm_MG_illumina_7008_11933_23519_93193'] = 'SR-VP_07_25_2022_A1_115cm_MG_illumina'
project_id_map['SRVP19_trench_A_40cm-2_viral_idba_scaffold_30523'] = 'SRVP19_trench_A_40cm-2_viral'
project_id_map['SR-VP_07_25_2022_A1_60cm_MG_illumina_5472_8855_31448'] = 'SR-VP_07_25_2022_A1_60cm_MG_illumina'
project_id_map['SR-VP_9_9_2021_56_4A_0_95m_scaffold_3886_5485'] = 'SR-VP_9_9_2021_56_4A_0.95m'


genome_id_map = {contig_id:genome_id for genome_id, contig_ids in genomes.items() for contig_id in contig_ids}
contigs_df['genome_id'] = contigs_df.contig_id.map(genome_id_map)
contigs_df = contigs_df[~contigs_df.genome_id.isnull()].copy()
contigs_df['project_id'] = contigs_df.genome_id.map(project_id_map)

reads_paths_df = pd.read_csv('../data/reads_paths.csv')
contigs_df = contigs_df.merge(reads_paths_df, on='project_id', how='left')
contigs_df.index = contigs_df.contig_id

In [32]:

biotite_dir = '/home/philippar/curation/'
ref_paths = list()

for genome_id, contig_ids in genomes.items():
    for i, row in enumerate(contigs_df[contigs_df.contig_id.isin(contig_ids)].itertuples()):
        contig_id =  f'scaffold_{i + 1}.fasta'

        contigs_df.loc[row.contig_id, 'ref_path'] = os.path.join(biotite_dir, genome_id, contig_id)
        contigs_df.loc[row.contig_id, 'genome_id'] = genome_id

        if not os.path.exists(os.path.join('../data/curation/', genome_id)):
            os.mkdir(os.path.join('../data/curation/', genome_id))
            
        with open(os.path.join('../data/curation/', genome_id, contig_id), 'w') as f:
            lines = [f'>{contig_id}']
            lines += [row.seq]
            f.write('\n'.join(lines))

In [33]:
def bbmap_get_command(ref_path, output_dir:str=None, forward_reads_path:str=None, reverse_reads_path:str=None):

    ref_id = os.path.basename(ref_path).replace('.fasta', '')
    output_path = os.path.join(output_dir, f'{ref_id}.bam')

    params = 'pigz=t unpigz=t ambiguous=random minid=0.96 idfilter=0.97 threads=64 out=stdout.sam editfilter=5'
    cmd = f'bbmap.sh {params} in1={forward_reads_path} in2={reverse_reads_path} ref={ref_path} nodisk | shrinksam | sambam > {output_path}'
    return cmd, output_path

In [34]:
for row in contigs_df.itertuples():
    try:
        output_dir = os.path.join(biotite_dir, row.genome_id)
        cmd, output_path = bbmap_get_command(row.ref_path, output_dir=output_dir, forward_reads_path=row.forward_reads_path, reverse_reads_path=row.reverse_reads_path)
        print(cmd)
    except:
        continue

bbmap.sh pigz=t unpigz=t ambiguous=random minid=0.96 idfilter=0.97 threads=64 out=stdout.sam editfilter=5 in1=/groups/banfield/sequences/2020/SRVP19_trench_A_40cm-2_viral/raw.d/SRVP19_trench_A_40cm-2_viral_trim_clean.PE.1.fastq.gz in2=/groups/banfield/sequences/2020/SRVP19_trench_A_40cm-2_viral/raw.d/SRVP19_trench_A_40cm-2_viral_trim_clean.PE.2.fastq.gz ref=/home/philippar/curation/SRVP19_trench_A_40cm-2_viral_idba_scaffold_30523/scaffold_1.fasta nodisk | shrinksam | sambam > /home/philippar/curation/SRVP19_trench_A_40cm-2_viral_idba_scaffold_30523/scaffold_1.bam
bbmap.sh pigz=t unpigz=t ambiguous=random minid=0.96 idfilter=0.97 threads=64 out=stdout.sam editfilter=5 in1=/groups/banfield/sequences/2021/SR-VP_9_9_2021_56_4A_0.95m/raw.d/SR-VP_9_9_2021_56_4A_0.95m_trim_clean.PE.1.fastq.gz in2=/groups/banfield/sequences/2021/SR-VP_9_9_2021_56_4A_0.95m/raw.d/SR-VP_9_9_2021_56_4A_0.95m_trim_clean.PE.2.fastq.gz ref=/home/philippar/curation/SR-VP_9_9_2021_56_4A_0_95m_scaffold_3886_5485/scaffol